# LLM-jp-4 8B BaseをGoogle Colabで動かす

`llm-jp/llm-jp-4-8b-base` を、Google Colab上でbitsandbytesによる4bit量子化を用いて実行します。

Baseモデルは、Instruct / Thinkingモデルとは役割が異なります。このNotebookでは違いが分かるように、**チャットモデルとしてではなく、与えた文章の続きを生成する基盤言語モデル**として扱います。

Instruct版からの主な変更点:

- `MODEL_ID` を `llm-jp/llm-jp-4-8b-base` に変更
- Harmony用の `parse_response()` を使用しない
- system / userメッセージによるチャットではなく、文字列promptを直接tokenize
- Gradioもチャット履歴型ではなく「Prompt → Continuation」のテキスト生成UIに変更

公式モデル:
https://huggingface.co/llm-jp/llm-jp-4-8b-base


In [1]:
# =========================================
# 実行環境の準備
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

%pip -q install -U "transformers==5.2.0" "accelerate>=1.13.0" bitsandbytes sentencepiece

import torch, transformers, accelerate
import bitsandbytes as bnb

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを有効にしてください。")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)
print("BF16 supported:", torch.cuda.is_bf16_supported())


GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-5eaffd76-7110-97fc-bcce-026eaebaf7ed)
Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 190.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 91.6 MB/s eta 0:00:00
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
PyTorch: 2.11.0+cu128
CUDA: 12.8
Transformers: 5.2.0
Accelerate: 1.14.0
bitsandbytes: 0.50.0
BF16 supported: True


In [2]:
# =========================================
# Google Driveとキャッシュ設定
# =========================================
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
CACHE_DIR = PROJECT_DIR / "Program" / "hf_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)
print("CACHE_DIR:", CACHE_DIR)


Mounted at /content/drive
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Program/hf_cache


In [3]:
# =========================================
# LLM-jp-4 8B Baseモデル
# =========================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "llm-jp/llm-jp-4-8b-base"

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=str(CACHE_DIR),
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)
model.eval()

INPUT_DEVICE = model.get_input_embeddings().weight.device

print("model loaded:", MODEL_ID)
print("compute dtype:", COMPUTE_DTYPE)
print(f"model memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")


config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/63.8k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.9MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

model loaded: llm-jp/llm-jp-4-8b-base
compute dtype: torch.bfloat16
model memory footprint: 6.25 GB


In [4]:
# =========================================
# Baseモデル用の文章続きを生成する関数
# =========================================
@torch.inference_mode()
def generate_continuation(
    prompt,
    max_new_tokens=128,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
):
    prompt = (prompt or "").strip()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(INPUT_DEVICE)

    generation_kwargs = {
        **inputs,
        "max_new_tokens": int(max_new_tokens),
        "do_sample": bool(do_sample),
        "use_cache": True,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        generation_kwargs["temperature"] = float(temperature)
        generation_kwargs["top_p"] = float(top_p)

    outputs = model.generate(**generation_kwargs)
    generated_ids = outputs[0, inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()


In [5]:
# =========================================
# 動作確認
# =========================================
prompt = "人工知能とは、"

print("Prompt:")
print(prompt)

print("\nGenerated continuation:")
print(generate_continuation(
    prompt,
    max_new_tokens=128,
    do_sample=True,
))


Prompt:
人工知能とは、

Generated continuation:
人間の知能を模倣するコンピュータシステムのことをいいます。
- 機械学習とは、人工知能の一分野であり、データから学習して予測や分類を行うアルゴリズムのことです。
- ディープラーニングとは、機械学習の一手法で、多層のニューラルネットワークを用いることで高度なパターン認識が可能な技術です。
- ニューラルネットワークとは、人間の脳の神経回路を模した計算モデルであり、複数のノード（ニューロン）が連結されて情報を処理します。
- 強化学習とは、エージェントが環境との相互作用を通じて報酬を最大化するための行動を学ぶ学習方式です。
- エージェントとは、問題解決や意思決定を行う主体となるソフトウェアやロボットのことです。
-


In [7]:
# =========================================
# Gradioを用いたBaseモデルのテキスト生成UI
# =========================================
import gradio as gr

def gr_generate(prompt):
    prompt = (prompt or "").strip()

    if not prompt:
        return ""

    return generate_continuation(
        prompt,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
    )


with gr.Blocks(title="LLM-jp-4 8B Base Text Completion") as demo:
    gr.Markdown(
        "## LLM-jp-4 8B Base Text Completion\n"
        "文章の書き出しを入力すると、その続きを生成します。"
    )

    prompt_box = gr.Textbox(
        value="人工知能とは、",
        label="Prompt",
        placeholder="文章の書き出しを入力してください。",
    )

    output_box = gr.Textbox(
        label="Generated continuation",
        lines=10,
    )

    with gr.Row():
        generate_btn = gr.Button("Generate", variant="primary")
        clear_btn = gr.Button("Clear")

    generate_btn.click(
        gr_generate,
        inputs=prompt_box,
        outputs=output_box,
        queue=False,
    )

    clear_btn.click(
        lambda: ("", ""),
        outputs=[prompt_box, output_box],
    )

print("WARNING: share=Trueで公開URLが作成されます。")

demo.launch(
    share=True,
    inline=True,
    debug=False,
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://28c6063e77389fe6cb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
